# Trabajo Práctico — Entrega 3

**Alumno:** Patricio Gerpe
**Materia:** Data Mining — Esp. en Explotación de Datos y Descubrimiento de Conocimiento (UBA Exactas)

Esta entrega avanza sobre el pipeline campeón de la Entrega 2 (E2-v4, Kaggle RMSE 93 151) incorporando:

- **Ingeniería de atributos** (Clase 7): nuevas variables derivadas de los datos propios.
- **Reducción de dimensionalidad** (Clase 8): PCA / VarianceThreshold / SelectKBest.
- **HP tuning** (habilitado en E3): `n_estimators` y `max_depth` del RandomForestRegressor.

Estructura del notebook:

1. Lectura de datos.
2. Limpieza y transformación (filtros E2 + outliers E2 + imputación E2 + **FE nuevo** + **dim reduction nuevo**).
3. Entrenamiento del modelo (RandomForestRegressor — sección no modificable).
4. Predicción sobre `a_predecir.csv` y override por *Hot Deck*.
5. Análisis de errores.

In [ ]:
import re
import sys
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Modo local: se asume que los datasets están en ./datasets")

In [ ]:
import os, json, hashlib, datetime, platform

ENTREGA     = os.environ.get("EXPERIMENT_ENTREGA", "entrega_3")
NOMBRE      = os.environ.get("EXPERIMENT_NAME",    "v1")
DESCRIPCION = os.environ.get(
    "EXPERIMENT_DESC",
    "E2 pipeline exact + HP sweep 3x3 (500/1000/1500 x 50/70/None)",
)

# HP: pueden venir de env (para v2+, fijados con el mejor de v1) o usar defaults.
# El modelo no cambia (RF); sólo estas dos perillas son modificables en E3.
n_estimators = int(os.environ.get("EXPERIMENT_N_ESTIMATORS", "500"))
_max_depth_env = os.environ.get("EXPERIMENT_MAX_DEPTH", "50")
max_depth = None if _max_depth_env == "None" else int(_max_depth_env)

EXPERIMENT_LOG = {
    "entrega":     ENTREGA,
    "nombre":      NOMBRE,
    "descripcion": DESCRIPCION,
    "timestamp":   datetime.datetime.now().isoformat(timespec="seconds"),
    "hostname":    platform.node(),
    "python":      platform.python_version(),
    "in_colab":    IN_COLAB,
    "params":      {},
    "data":        {},
    "metricas":    {},
    "kaggle": {"rmse": None, "submitted_at": None, "leaderboard_position": None},
}
print(f"Experimento: {ENTREGA}/{NOMBRE}")
print(f"Descripción: {DESCRIPCION}")
print(f"HP iniciales: n_estimators={n_estimators}, max_depth={max_depth}")

## 0. Lectura de datos

Levantamos `entrenamiento.db` (train, ~1.29 M filas) y `a_predecir.csv` (test, 13 471 filas).

In [ ]:
DIR = "/content/drive/MyDrive/datos/propiedades" if IN_COLAB else "datasets"

In [ ]:
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")
df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col="id")
df_ap  = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")

df_ent["__src__"] = "train"
df_ap["__src__"]  = "test"

print(f"train: {df_ent.shape}  |  test: {df_ap.shape}")

In [ ]:
df_ent.shape, df_ap.shape

In [ ]:
df_ent.columns.tolist()

## 1. Entender los datos (AID)

Recap de E2: el universo a modelar es `venta` en `dolares` en CABA, ~122 k filas de train.

In [ ]:
nulos = pd.concat(
    [df_ent.isna().mean().rename("train_%"), df_ap.isna().mean().rename("test_%")],
    axis=1,
).sort_values("train_%", ascending=False).round(3)
nulos

## 2. Limpiar y transformar los datos (DM)

## 2.1. Filtrado de datos

Idéntico a E2-v4: CABA ampliado (location_1 estricto + location_2/3 para recuperar mal-etiquetadas),
operación venta, moneda dólares, property_type ∈ {departamento, casa, ph, cochera},
price ∈ [5 000, 3 000 000].

In [ ]:
CABA_L1 = {"Capital Federal", "Ciudad Autónoma de Buenos Aires"}
PROP_OK = {"departamento", "departamentos", "casa", "casas", "ph", "cochera"}

caba_loc2 = set(df_ap["location_2"].dropna().unique())
caba_loc3 = set(df_ap["location_3"].dropna().unique())

mask_caba = (
    df_ent["location_1"].isin(CABA_L1)
    | (
        df_ent["location_1"].eq("Buenos Aires")
        & (df_ent["location_2"].isin(caba_loc2) | df_ent["location_3"].isin(caba_loc3))
    )
)
mask = (
    df_ent["operation_type"].eq("venta")
    & df_ent["currency_type"].eq("dolares")
    & mask_caba
    & df_ent["property_type"].isin(PROP_OK)
    & df_ent["price"].notna()
    & df_ent["price"].between(5_000, 3_000_000)
)
df_ent = df_ent.loc[mask].copy()

df_ent["property_type"] = df_ent["property_type"].replace({"departamentos": "departamento", "casas": "casa"})
df_ap["property_type"]  = df_ap["property_type"].replace({"departamentos": "departamento", "casas": "casa"})

print(f"train luego de filtrar: {df_ent.shape}")
print(df_ent["property_type"].value_counts())

EXPERIMENT_LOG["params"]["filtros"] = {
    "caba_l1_estricto": sorted(CABA_L1),
    "caba_l1_ampliado_via_loc2_loc3": True,
    "property_type": sorted(PROP_OK),
    "operation_type": "venta",
    "currency_type": "dolares",
    "price_min": 5000,
    "price_max": 3_000_000,
}
EXPERIMENT_LOG["data"]["train_post_filtro"] = list(df_ent.shape)

In [ ]:
def parse_features(serie: pd.Series) -> pd.DataFrame:
    AMENITIES = [
        "balcon", "garage", "cochera", "pileta", "parrilla",
        "aire acondicionado", "calefaccion", "gas natural",
        "internet", "seguridad", "alarma", "gimnasio", "jardin",
        "cuarto de servicio", "cocina equipada", "bodega",
    ]
    s = serie.fillna("").astype(str).str.lower()
    for src, dst in [("á","a"),("é","e"),("í","i"),("ó","o"),("ú","u"),("ñ","n")]:
        s = s.str.replace(src, dst, regex=False)

    out = pd.DataFrame(index=serie.index)
    out["n_dormitorios"] = s.str.extract(r"(\d+)\s*dormitor", expand=False).astype(float)
    out["n_banos"]       = s.str.extract(r"(\d+)\s*bano",     expand=False).astype(float)
    out["m2"]            = s.str.extract(r"(\d+)\s*m[²2]",    expand=False).astype(float)
    for a in AMENITIES:
        out[f"f_{a.replace(' ', '_')}"] = s.str.contains(a, regex=False).astype(int)
    return out


def parse_features_dual(features_col: pd.Series, description_col: pd.Series):
    """v2+: parsea features primero; si NaN en m2/dormitorios/banos, recurre a description.
    Agrega rooms (ambientes) desde description."""
    out = parse_features(features_col)
    desc_parsed = parse_features(description_col)

    rescued = {}
    for col in ["m2", "n_dormitorios", "n_banos"]:
        mask = out[col].isna() & desc_parsed[col].notna()
        rescued[col] = int(mask.sum())
        out.loc[mask, col] = desc_parsed.loc[mask, col]

    # rooms: patron "X amb" desde description
    s_desc = description_col.fillna("").astype(str).str.lower()
    for src, dst in [("á","a"),("é","e"),("í","i"),("ó","o"),("ú","u"),("ñ","n")]:
        s_desc = s_desc.str.replace(src, dst, regex=False)
    out["rooms"] = s_desc.str.extract(r"(\d+)\s*amb", expand=False).astype(float)

    return out, rescued


def extra_attrs(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["len_descripcion"] = df["description"].fillna("").str.len()
    out["n_features"]      = df["features"].fillna("").str.count(";")
    out["barrio"]          = df["location_3"].fillna("desconocido").astype(str)
    return out


feat_ent, rescued_ent = parse_features_dual(df_ent["features"], df_ent["description"])
feat_ap,  rescued_ap  = parse_features_dual(df_ap["features"],  df_ap["description"])

df_ent = pd.concat([df_ent, feat_ent, extra_attrs(df_ent)], axis=1)
df_ap  = pd.concat([df_ap,  feat_ap,  extra_attrs(df_ap)],  axis=1)

print("Nuevas columnas:", feat_ent.columns.tolist())
print(f"[parseo dual train] m2={rescued_ent['m2']:,}  dormitorios={rescued_ent['n_dormitorios']:,}  banos={rescued_ent['n_banos']:,}")
print(f"[parseo dual test ] m2={rescued_ap['m2']:,}  dormitorios={rescued_ap['n_dormitorios']:,}  banos={rescued_ap['n_banos']:,}")
df_ent[["n_dormitorios", "n_banos", "m2", "rooms"]].describe()

In [ ]:
print("=== Percentiles de m2, n_dormitorios, n_banos ===")
percs = [0.001, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
display(df_ent[["m2", "n_dormitorios", "n_banos"]].quantile(percs).round(1).T)

## 2.2. Tratamiento de valores atípicos

Idéntico a E2-v4: winsorización de m2/dormitorios/baños + IsolationForest multivariado (contaminación 1 %, solo train).

In [ ]:
for df_ in (df_ent, df_ap):
    df_.loc[~df_["m2"].between(10, 1500), "m2"] = np.nan
    df_.loc[~df_["n_dormitorios"].between(0, 15), "n_dormitorios"] = np.nan
    df_.loc[~df_["n_banos"].between(0, 15),       "n_banos"]       = np.nan

iso_cols = ["price", "m2", "n_dormitorios", "n_banos"]
mask_complete = df_ent[iso_cols].notna().all(axis=1)
X_iso = df_ent.loc[mask_complete, iso_cols].astype(float)

scaler = StandardScaler()
X_iso_sc = scaler.fit_transform(X_iso)

iso = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
y_iso = iso.fit_predict(X_iso_sc)

ids_outliers = X_iso.index[y_iso == -1]
print(f"IsolationForest: {len(ids_outliers)} filas marcadas.")
df_ent = df_ent.drop(index=ids_outliers)
print(f"train tras remover outliers: {df_ent.shape}")

EXPERIMENT_LOG["params"]["outlier_caps"] = {
    "m2": [10, 1500], "n_dormitorios": [0, 15], "n_banos": [0, 15],
}
EXPERIMENT_LOG["params"]["isolation_forest"] = {
    "contamination": 0.01, "cols": iso_cols, "scaled": "StandardScaler",
}
EXPERIMENT_LOG["data"]["outliers_if_eliminados"] = int(len(ids_outliers))
EXPERIMENT_LOG["data"]["train_post_outliers"]    = list(df_ent.shape)

In [ ]:
# --- §2.2b v10: Dedup price tests — DESACTIVADO (v10 cancelado: hurts RF) ---
# v10 probó colapsar propiedades repetidas a mediana de precio.
# Resultado: CV5 parcial fold 2/15 = 118 574 (+7 414 vs campeón v8).
# Aprendizaje: RF se beneficia del volumen de datos; eliminar ~10% del train
# por "pureza de labels" es contraproducente — la redundancia aporta cobertura.
print("[v10 dedup] DESACTIVADO — experimento cancelado por early stopping fold 2.")

In [ ]:
# (celda vacía: tratamiento de outliers completo en celda anterior)

## 2.3. Imputación de valores perdidos

Idéntico a E2-v4: mediana global para numéricas, moda para categóricas, marcador `m2_was_na`.

In [ ]:
for df_ in (df_ent, df_ap):
    df_["m2_was_na"] = df_["m2"].isna().astype(int)

NUM_MED  = ["m2", "lat", "lon", "n_dormitorios", "n_banos", "len_descripcion", "n_features", "rooms"]
CAT_MODE = ["property_type", "barrio"]

imp_med = SimpleImputer(strategy="median")
imp_med.fit(df_ent[NUM_MED])
df_ent[NUM_MED] = imp_med.transform(df_ent[NUM_MED])
df_ap[NUM_MED]  = imp_med.transform(df_ap[NUM_MED])

flag_cols = [c for c in df_ent.columns if c.startswith("f_")]
df_ent[flag_cols] = df_ent[flag_cols].fillna(0).astype(int)
df_ap[flag_cols]  = df_ap[flag_cols].fillna(0).astype(int)

imp_mod = SimpleImputer(strategy="most_frequent")
imp_mod.fit(df_ent[CAT_MODE])
df_ent[CAT_MODE] = imp_mod.transform(df_ent[CAT_MODE])
df_ap[CAT_MODE]  = imp_mod.transform(df_ap[CAT_MODE])

EXPERIMENT_LOG["params"]["imputacion"] = {
    "numericas": {c: "median (global, fit on train)" for c in NUM_MED},
    "categoricas": {c: "most_frequent (fit on train)" for c in CAT_MODE},
    "marcador_m2_was_na": True,
}
print("Faltantes restantes (top 10):")
print(df_ent.isna().sum().sort_values(ascending=False).head(10))

## 2.4. Creación de nuevos atributos (FE — NUEVO E3)

**Placeholder v1** — pipeline idéntico a E2. Se completa con versiones sucesivas:

| versión | cambio |
|---|---|
| **v2** | parseo dual (`features` + `description`): rescata m2 (8.86 %), dormitorios (3.07 %), baños (1.86 %); feature `rooms` (ambientes). |
| v3 | log-transforms de features (DESCARTADO — RF invariante a transforms monótonos). |
| **v4** | barrio cleanup (cascade Hot Deck → KNN → "desconocido") + colapso de barrios escasos (≤ 10 obs). |
| **v5** | distancias a centros de referencia derivados del propio dataset + ratios m2_per_room/bano + BallTree density k=10. |
| v6 | log(price) target transform (CANCELADO — RF en log-space predice mediana geométrica, no media aritmética; RMSE original empeora). |
| **v7** | KNN imputation para m2 usando lat/lon como proximidad geográfica (k=5, fit only on train): reemplaza mediana global para filas donde lat/lon están disponibles. |

Regla de leakage: **no** derivar estadísticos del precio por barrio (target encoding) — produce distributional shift como en E2-v2/v3.

In [ ]:
# --- §2.4 Feature Engineering — v2: parseo dual (features + description fallback) ---
# El parseo dual ya corrio en la celda de parse_features_dual (arriba).
# Aqui registramos las estadisticas de rescate en el log.

n_train = len(df_ent)
print(f"[FE v2] parseo dual rescato en train:")
print(f"  m2          : {rescued_ent['m2']:>6,} filas ({rescued_ent['m2']/n_train*100:.2f}%)")
print(f"  dormitorios : {rescued_ent['n_dormitorios']:>6,} filas ({rescued_ent['n_dormitorios']/n_train*100:.2f}%)")
print(f"  banos       : {rescued_ent['n_banos']:>6,} filas ({rescued_ent['n_banos']/n_train*100:.2f}%)")
print(f"  rooms (no-nulos pre-impute): {feat_ent['rooms'].notna().sum():,}")

EXPERIMENT_LOG["params"]["fe_version"] = "v12-parseo-dual+barrio-cleanup+scores+tfidf-svd"
EXPERIMENT_LOG["params"]["fe_parseo_dual"] = {
    "descripcion": "fallback description cuando features da NaN en m2/dormitorios/banos",
    "nueva_feature": "rooms (ambientes desde description, patron X amb)",
    "rescatados_train": rescued_ent,
    "rescatados_test":  rescued_ap,
}

In [ ]:
# --- §2.4 Feature Engineering — v4: barrio cleanup (cascade HotDeck -> KNN -> desconocido) ---
#
# Motivacion (error analysis):
#   71.25% del RMSE viene de propiedades con "barrio desconocido" (etiquetas ciudad-nivel
#   como "Ciudad Autonoma de Buenos Aires" en lugar de un barrio especifico).
#   Cascada: 1) Hot Deck por descripcion normalizada, 2) KNN(lat/lon k=5), 3) "desconocido".
#   Colapso: barrios con <= 10 obs en train -> "barrio_raro" (reduce cardinalidad).
#
# Nota: se ejecuta DESPUES de imputacion (lat/lon ya rellenados con mediana).
# Para filas con lat/lon originalmente NaN, KNN usa coords de mediana (centro CABA) ->
# asigna barrio del centro, mejor que "desconocido" en la mayoria de casos.

import unicodedata as _ucd2, re as _re2
from sklearn.neighbors import KNeighborsClassifier as _KNC

_BASURA_LABELS_BARRIO = {
    "ciudad autonoma de buenos aires",
    "ciudad de buenos aires",
    "capital federal",
    "buenos aires",
}

def _norm_barrio_str(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = _ucd2.normalize("NFKD", str(s))
    s = "".join(c for c in s if not _ucd2.combining(c))
    return _re2.sub(r'\s+', ' ', s.lower().strip())

def _is_basura_barrio(v):
    return _norm_barrio_str(v) in (_BASURA_LABELS_BARRIO | {""})

def _norm_desc_barrio(s):
    """Normalizar descripcion para busqueda en HotDeck (identico a _norm_desc de §2.6)."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = _ucd2.normalize("NFKD", str(s))
    s = "".join(c for c in s if not _ucd2.combining(c))
    s = _re2.sub(r'[^0-9a-z\s]', ' ', s.lower())
    s = _re2.sub(r'\s+', ' ', s).strip()
    return s if s else None

# --- Build Hot Deck dict: desc_norm -> barrio (primer match en train con barrio limpio) ---
_clean_mask_bc = ~df_ent["barrio"].apply(_is_basura_barrio)
_hd_barrio_dict = {}
if "description" in df_ent.columns:
    _tmp_bc = df_ent.loc[_clean_mask_bc, ["description", "barrio"]].copy()
    _tmp_bc["_nd"] = _tmp_bc["description"].apply(_norm_desc_barrio)
    for nd_, barrio_ in zip(_tmp_bc["_nd"], _tmp_bc["barrio"]):
        if nd_ and nd_ not in _hd_barrio_dict:
            _hd_barrio_dict[nd_] = barrio_
    del _tmp_bc
print(f"[FE v4] HotDeck dict barrio: {len(_hd_barrio_dict):,} entradas desc->barrio")

# --- Fit KNN on clean train rows with non-NaN lat/lon ---
_knn_train_bc = df_ent[
    _clean_mask_bc & df_ent["lat"].notna() & df_ent["lon"].notna()
].copy()
_knn_barrio = _KNC(n_neighbors=5, metric="euclidean", n_jobs=-1)
_knn_barrio.fit(_knn_train_bc[["lat", "lon"]].values, _knn_train_bc["barrio"].values)
print(f"[FE v4] KNN barrio entrenado: {len(_knn_train_bc):,} filas train limpias con lat/lon")

# --- Apply cascade to both dataframes ---
def _apply_barrio_cleanup(df, is_test=False):
    df = df.copy()
    basura = df["barrio"].apply(_is_basura_barrio)
    n_total = basura.sum()

    # Step 1: Hot Deck por descripcion normalizada
    n_hd = 0
    if "description" in df.columns and _hd_barrio_dict:
        nd_series = df.loc[basura, "description"].apply(_norm_desc_barrio)
        resolved = nd_series.map(_hd_barrio_dict).dropna()
        df.loc[resolved.index, "barrio"] = resolved
        n_hd = len(resolved)
    basura = df["barrio"].apply(_is_basura_barrio)

    # Step 2: KNN para restantes con lat/lon disponible
    knn_cand = basura & df["lat"].notna() & df["lon"].notna()
    n_knn = knn_cand.sum()
    if n_knn > 0:
        preds = _knn_barrio.predict(df.loc[knn_cand, ["lat", "lon"]].values)
        df.loc[knn_cand, "barrio"] = preds
    basura = df["barrio"].apply(_is_basura_barrio)

    # Step 3: desconocido para el resto
    n_desconocido = basura.sum()
    df.loc[basura, "barrio"] = "desconocido"

    lbl = "test" if is_test else "train"
    print(f"[FE v4] barrio {lbl}: basura_total={n_total} -> HotDeck={n_hd} KNN={n_knn} desconocido={n_desconocido}")
    return df

df_ent = _apply_barrio_cleanup(df_ent, is_test=False)
df_ap  = _apply_barrio_cleanup(df_ap,  is_test=True)

# --- Colapsar barrios escasos (<=10 obs en train) -> "barrio_raro" ---
_barrio_counts = df_ent["barrio"].value_counts()
_barrios_escasos = set(_barrio_counts[_barrio_counts <= 10].index)
df_ent["barrio"] = df_ent["barrio"].apply(lambda x: "barrio_raro" if x in _barrios_escasos else x)
df_ap ["barrio"] = df_ap ["barrio"].apply(lambda x: "barrio_raro" if x in _barrios_escasos else x)
n_esc_train = df_ent["barrio"].eq("barrio_raro").sum()
print(f"[FE v4] {len(_barrios_escasos)} barrios escasos (<=10 obs) -> 'barrio_raro' ({n_esc_train} filas train)")
print(f"[FE v4] barrios unicos post-cleanup: train={df_ent['barrio'].nunique()} test={df_ap['barrio'].nunique()}")

EXPERIMENT_LOG["params"]["fe_barrio_cleanup"] = {
    "descripcion": "cascade HotDeck(desc_norm) -> KNN(lat/lon, k=5) -> desconocido; escasos<=10 -> barrio_raro",
    "basura_labels": sorted(_BASURA_LABELS_BARRIO),
    "knn_n_train": len(_knn_train_bc),
    "hd_dict_size": len(_hd_barrio_dict),
    "n_barrios_escasos": len(_barrios_escasos),
}


In [ ]:
# --- §2.4 Feature Engineering — v5: distancias a centros + ratios + densidad ---
#
# (a) m2_per_room / m2_per_bano: ratio de espaciosidad — nuevo split en 1 var vs 2 crudas.
# (b) Distancias euclideas (* 111 km) a los N barrios mas frecuentes en train:
#     captura estructura radial de precios de CABA mejor que lat/lon crudos.
# (c) BallTree density: distancia media a los k=10 vecinos mas cercanos en train
#     como proxy de densidad urbana (diferencia zona residencial vs comercial).
# Todo fit en train, transform en ambos. Sin leakage, sin APIs.

import numpy as np, re as _re5, unicodedata as _ucd5
from sklearn.neighbors import BallTree as _BallTree

# ── (a) Ratios de espaciosidad ──────────────────────────────────────────────
for df_ in (df_ent, df_ap):
    df_["m2_per_room"] = df_["m2"] / (df_["n_dormitorios"].clip(lower=0) + 1)
    df_["m2_per_bano"] = df_["m2"] / (df_["n_banos"].clip(lower=0) + 1)
print(f"[FE v5] m2_per_room y m2_per_bano agregados")

# ── (b) Distancias a centros de referencia ──────────────────────────────────
_MIN_OBS_CENTRO = 50
_N_CENTROS      = 10

def _slug5(s):
    s = _ucd5.normalize("NFKD", str(s))
    s = "".join(c for c in s if not _ucd5.combining(c))
    return _re5.sub(r"_+", "_", _re5.sub(r"[^a-z0-9]", "_", s.lower())).strip("_")[:18]

_bc_valid = df_ent[
    (~df_ent["barrio"].isin({"desconocido", "barrio_raro"})) &
    df_ent["lat"].notna() & df_ent["lon"].notna()
]
_centros = (
    _bc_valid.groupby("barrio")
    .agg(lat_med=("lat","median"), lon_med=("lon","median"), n=("lat","count"))
    .reset_index()
    .query("n >= @_MIN_OBS_CENTRO")
    .sort_values("n", ascending=False)
    .head(_N_CENTROS)
    .reset_index(drop=True)
)
print(f"[FE v5] {len(_centros)} centros de referencia:")
for _, r in _centros.iterrows():
    print(f"  {r['barrio']}: n={r['n']}")

_dist_cols_v5 = []
for _, centro in _centros.iterrows():
    col = "dist_" + _slug5(centro["barrio"])
    dist_tr = np.sqrt((df_ent["lat"] - centro["lat_med"])**2 + (df_ent["lon"] - centro["lon_med"])**2) * 111.0
    _med = float(dist_tr.median())
    df_ent[col] = dist_tr.fillna(_med)
    dist_te = np.sqrt((df_ap["lat"] - centro["lat_med"])**2 + (df_ap["lon"] - centro["lon_med"])**2) * 111.0
    df_ap[col]  = dist_te.fillna(_med)
    _dist_cols_v5.append(col)
print(f"[FE v5] distancias agregadas: {_dist_cols_v5}")

# ── (c) BallTree density (k=10 vecinos mas cercanos, distancia media) ────────
_K_DENSITY = 10
_bt_train = df_ent[df_ent["lat"].notna() & df_ent["lon"].notna()][["lat","lon"]].copy()
_coords_train_rad = np.radians(_bt_train.values)
_bt = _BallTree(_coords_train_rad, metric="haversine")

def _density_col(df_):
    has_coords = df_["lat"].notna() & df_["lon"].notna()
    result = np.full(len(df_), np.nan)
    if has_coords.sum() > 0:
        coords_rad = np.radians(df_.loc[has_coords, ["lat","lon"]].values)
        dists, _ = _bt.query(coords_rad, k=_K_DENSITY + 1)  # +1 porque train incluye el punto mismo
        result[has_coords.values] = dists[:, 1:].mean(axis=1) * 6371.0  # radianes -> km
    return result

_dens_tr = _density_col(df_ent)
_dens_med = float(np.nanmedian(_dens_tr))
df_ent["density_k10"] = np.where(np.isnan(_dens_tr), _dens_med, _dens_tr)
_dens_te = _density_col(df_ap)
df_ap["density_k10"]  = np.where(np.isnan(_dens_te), _dens_med, _dens_te)
print(f"[FE v5] density_k10: media={df_ent['density_k10'].mean():.3f} km, std={df_ent['density_k10'].std():.3f}")

EXPERIMENT_LOG["params"]["fe_distancias"] = {
    "descripcion": "m2_per_room/bano + distancias eucl. a top-10 barrios + BallTree density k=10",
    "n_centros": len(_centros),
    "centros": _centros["barrio"].tolist(),
    "dist_cols": _dist_cols_v5,
    "density_col": "density_k10",
}


In [ ]:
# --- §2.4 Feature Engineering — v11: scores temáticos de amenities (Clase 07) ---
# Agrupa las 16 features binarias en 5 scores semánticos.
# Motivación: el score pre-computado captura el efecto aditivo directamente,
# reduciendo la profundidad de árbol necesaria para detectar la combinación.
# Las 16 binarias originales se conservan para que SelectKBest elija las mejores.

SCORE_DEFS = {
    "score_lujo":            ["f_pileta", "f_gimnasio", "f_parrilla", "f_jardin"],
    "score_servicios":       ["f_aire_acondicionado", "f_calefaccion", "f_gas_natural", "f_internet"],
    "score_estacionamiento": ["f_cochera", "f_garage"],
    "score_espacios_extra":  ["f_balcon", "f_bodega", "f_cuarto_de_servicio"],
    "score_seguridad":       ["f_seguridad", "f_alarma"],
}

for df_ in (df_ent, df_ap):
    for score_name, components in SCORE_DEFS.items():
        df_[score_name] = df_[components].sum(axis=1)

print("[FE v11] scores temáticos (Clase 07):")
for score_name, components in SCORE_DEFS.items():
    print(f"  {score_name:30s}: media={df_ent[score_name].mean():.3f}  max={df_ent[score_name].max()}")

EXPERIMENT_LOG["params"]["fe_amenity_scores"] = {
    "descripcion": "5 scores temáticos sumando binarias de amenities (Clase 07 FE)",
    "scores": {k: v for k, v in SCORE_DEFS.items()},
}

## 2.5. Reducción de dimensionalidad (NUEVO E3)

| versión | técnica |
|---|---|
| v1–v7 | placeholder (sin cambios) |
| **v8** | TF-IDF + TruncatedSVD (20 comp, vocab 5k) — campeón previo. |
| **v9** | **TF-IDF + TruncatedSVD (50 comp, vocab 10k)** — escala v8: más componentes + vocabulario más amplio. |

 siempre en train,  en ambos (sin leakage).

In [ ]:
# --- §2.5 Reducción de dimensionalidad — v11: SelectKBest — DESACTIVADO en v12 ---
# v11 eliminó f_calefaccion, f_gas_natural, f_internet via SelectKBest(mutual_info, k=10).
# Esas features tienen más señal en test que en train -> distribution shift Kaggle +736 vs v8.
# v12 conserva las 16 binarias originales + 5 scores temáticos para aislar el efecto.
print("[§2.5 v12] SelectKBest DESACTIVADO — se conservan todas las amenities binarias")


In [ ]:
# --- §2.5 Reducción de dimensionalidad — v8/v11/v12: TF-IDF + TruncatedSVD sobre description ---
# v12 mantiene v8/v11 settings (N_SVD=20, 5k vocab, bigramas) — combinado con SelectKBest de arriba.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

N_SVD = 20

desc_tr = df_ent["description"].fillna("").astype(str)
desc_ap = df_ap["description"].fillna("").astype(str)

tfidf_v = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True,
    strip_accents="unicode",
    lowercase=True,
)
svd_v = TruncatedSVD(n_components=N_SVD, random_state=42)

X_tfidf_tr = tfidf_v.fit_transform(desc_tr)
X_svd_tr   = svd_v.fit_transform(X_tfidf_tr)
X_tfidf_ap = tfidf_v.transform(desc_ap)
X_svd_ap   = svd_v.transform(X_tfidf_ap)

svd_cols = [f"svd_desc_{i}" for i in range(N_SVD)]
df_ent[svd_cols] = X_svd_tr
df_ap[svd_cols]  = X_svd_ap

var_explained = svd_v.explained_variance_ratio_.sum()
print(f"[§2.5 v11] TF-IDF+SVD: {N_SVD} componentes | varianza explicada: {var_explained:.1%}")

EXPERIMENT_LOG["params"]["dim_reduction_version"] = "v12-scores-temáticos+tfidf-svd-20comp-5kvocab"
EXPERIMENT_LOG["params"]["dim_reduction_tfidf_svd"] = {
    "tecnica": "TF-IDF + TruncatedSVD",
    "n_svd_components": N_SVD,
    "tfidf_max_features": 5000,
    "varianza_explicada": round(float(var_explained), 4),
}

## 2.6. Hot Deck por descripción duplicada

Construimos el diccionario `id_test → precio_mediano` buscando descripciones normalizadas idénticas
entre train y test. La aplicación del override se hace en §4.

In [ ]:
import unicodedata

def _norm_desc(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = str(s)
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = re.sub(r'[^0-9a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s if len(s) >= 20 else None

_eng = sqlite3.connect(f"{DIR}/entrenamiento.db")
_train_full = pd.read_sql(
    '''
    SELECT id, description, price
    FROM entrenamiento
    WHERE operation_type = 'venta'
      AND currency_type  = 'dolares'
      AND price IS NOT NULL
      AND description    IS NOT NULL
    ''',
    _eng,
)
_train_full["desc_norm"] = _train_full["description"].map(_norm_desc)

price_by_desc  = _train_full.dropna(subset=["desc_norm"]).groupby("desc_norm")["price"].median()
test_desc_norm = df_ap["description"].map(_norm_desc).dropna()
hotdeck_dict   = test_desc_norm.map(price_by_desc).dropna().to_dict()

cov_hd = len(hotdeck_dict) / len(df_ap) * 100
print(f"Hot Deck: {len(hotdeck_dict)} filas → {cov_hd:.2f}% del test")

EXPERIMENT_LOG["params"]["hotdeck"] = {
    "key": "description_normalizada",
    "normalizacion": "lower + sin_acentos + sin_puntuacion + colapsar_espacios + min_20_chars",
    "agg": "median",
    "fuente": "train completo (venta + USD)",
}
EXPERIMENT_LOG["data"]["hotdeck_overrides"]    = int(len(hotdeck_dict))
EXPERIMENT_LOG["data"]["hotdeck_coverage_pct"] = round(cov_hd, 2)

del _train_full, price_by_desc, test_desc_norm

In [ ]:
for col in ["barrio", "property_type"]:
    codes, uniques = pd.factorize(df_ent[col].astype(str))
    df_ent[f"{col}_id"] = codes
    mapping = {v: i for i, v in enumerate(uniques)}
    df_ap[f"{col}_id"] = df_ap[col].astype(str).map(mapping).fillna(-1).astype(int)

print("ids únicos:", df_ent[["barrio_id", "property_type_id"]].nunique().to_dict())

In [ ]:
# Target encoding por barrio: DESACTIVADO (leakage detectado en E2-v2/v3).
print("v1: NO se agregan features de target encoding por barrio (decisión post-leakage E2).")
EXPERIMENT_LOG["params"]["target_encoding_barrio"] = "DESACTIVADO (leakage E2-v2/v3)"

In [ ]:
# Guardamos los índices del split temporal 80/20 ANTES de select_dtypes
# (que descarta publication_date por ser no-numérica).
if "publication_date" in df_ent.columns:
    _sorted_dates = df_ent["publication_date"].sort_values()
    _n_hold = int(len(_sorted_dates) * 0.2)
    temporal_test_idx  = _sorted_dates.index[-_n_hold:]
    temporal_train_idx = _sorted_dates.index[:-_n_hold]
    print(f"Split temporal 80/20: {len(temporal_train_idx):,} train | {len(temporal_test_idx):,} holdout")
    print(f"  Rango temporal holdout: {_sorted_dates.iloc[-_n_hold]} → {_sorted_dates.iloc[-1]}")
else:
    temporal_test_idx  = None
    temporal_train_idx = None
    print("publication_date no disponible; holdout temporal desactivado.")

## 3. Entrenamiento del modelos (AA)- ⛔⛔⛔ NO CAMBIAR EL MODELO ⛔⛔⛔

El modelo es `RandomForestRegressor`. Los HP `n_estimators` y `max_depth` son los únicos parámetros modificables en E3.

In [ ]:
# La creación de modelos requiere que todo el dataframe sea numérico
# Me quedo con las columnas numéricas solamente
df_ent = df_ent.select_dtypes('number')

X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

In [ ]:
# --- §3 pre: target transform ---
# USE_LOG_TARGET = False (v11+/v12): antipatrón confirmado en v6.
# RF en log(y) minimiza RMSE en log-space (≈MAPE), predice mediana geométrica.
# Al invertir con exp(), el sesgo en escala USD original empeora el RMSE.

USE_LOG_TARGET = False
y_orig = y.copy()
y_fit  = np.log(y) if USE_LOG_TARGET else y

print(f"[v12] log-target: {USE_LOG_TARGET} (desactivado — antipatrón v6)")

In [ ]:
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(
    X, y_orig, test_size=0.2, random_state=42
)
y_fit_train = y_fit.loc[X_train.index]

# Definimos el valor de los hiperparámetros a usar por el modelo
# (pueden venir de env o del sweep de §3.1)

# Creamos el modelo a entrenar -- NO CAMBIAR EL MODELO
reg = sk.ensemble.RandomForestRegressor(
    n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42
)
_ = reg.fit(X_train, y_fit_train)

_pred_tr_raw = reg.predict(X_train)
_pred_te_raw = reg.predict(X_test)
if USE_LOG_TARGET:
    _pred_tr = np.exp(_pred_tr_raw)
    _pred_te = np.exp(_pred_te_raw)
else:
    _pred_tr = _pred_tr_raw
    _pred_te = _pred_te_raw

score_train = sk.metrics.root_mean_squared_error(y_train, _pred_tr)
score_test  = sk.metrics.root_mean_squared_error(y_test,  _pred_te)

print(f"{n_estimators=} -- {max_depth=} --> {score_train=:.2f} - {score_test=:.2f}")

In [ ]:
EXPERIMENT_LOG["params"]["modelo"] = {
    "tipo": "RandomForestRegressor",
    "n_estimators": int(n_estimators),
    "max_depth": str(max_depth),
    "random_state": 42,
    "split": "train_test_split(test_size=0.2, random_state=42)",
}
EXPERIMENT_LOG["metricas"]["rmse_train"]   = float(score_train)
EXPERIMENT_LOG["metricas"]["rmse_holdout"] = float(score_test)
EXPERIMENT_LOG["metricas"]["n_features"]   = int(X.shape[1])
EXPERIMENT_LOG["metricas"]["features"]     = X.columns.tolist()

In [ ]:
import sys as _sys, time
import numpy as _np

_sys.path.insert(0, str(__import__('pathlib').Path('.').resolve()))
from entregas._resumable import ResumableRunner

# CV5 multi-seed: 3 seeds x 5 folds = 15 fits, con checkpoint por fold.
# Si el proceso cae a mitad, la proxima corrida retoma desde el fold pendiente.
# RMSE siempre en USD original (escala comparable con versiones anteriores).
_SEEDS = [0, 1, 2]
_N_SPLITS = 5
_grid = [{"seed": s, "fold": f} for s in _SEEDS for f in range(_N_SPLITS)]

def _cv5_run_one(combo):
    _s, _f = combo["seed"], combo["fold"]
    _kf = sk.model_selection.KFold(n_splits=_N_SPLITS, shuffle=True, random_state=_s)
    _tr, _te = list(_kf.split(X, y_orig))[_f]
    _rf = sk.ensemble.RandomForestRegressor(
        n_estimators=n_estimators, max_depth=max_depth,
        n_jobs=-1, random_state=42,
    )
    _rf.fit(X.iloc[_tr], y_fit.iloc[_tr])
    _pred_raw = _rf.predict(X.iloc[_te])
    _pred = _np.exp(_pred_raw) if USE_LOG_TARGET else _pred_raw
    return {"rmse": float(sk.metrics.root_mean_squared_error(y_orig.iloc[_te], _pred))}

_md_str = str(max_depth)
_ckpt = f"entregas/{ENTREGA}/cv5_{NOMBRE}_n{n_estimators}_d{_md_str}.jsonl"

_t0 = time.time()
print(f"[CV5-multiseed] n_estimators={n_estimators}, max_depth={max_depth} | {len(_grid)} fits")
print(f"[CV5-multiseed] checkpoint: {_ckpt}")

_runner = ResumableRunner(
    progress_path=_ckpt,
    grid=_grid,
    run_fn=_cv5_run_one,
    key_fn=lambda c: f"s{c['seed']}_f{c['fold']}",
    log_every=1,
)
_runner.run()

_df_cv = _runner.results_df()
_cv5_all = _df_cv["rmse"].dropna().tolist()
_cv_mean = float(_np.mean(_cv5_all))
_cv_std  = float(_np.std(_cv5_all))
print(f"[CV5-multiseed] media: {_cv_mean:,.2f}  std: {_cv_std:,.2f}  ({time.time()-_t0:.0f}s)")

EXPERIMENT_LOG["metricas"]["rmse_cv5_folds"] = _cv5_all
EXPERIMENT_LOG["metricas"]["rmse_cv5_mean"]  = _cv_mean
EXPERIMENT_LOG["metricas"]["rmse_cv5_std"]   = _cv_std
EXPERIMENT_LOG["params"]["validacion_primaria"] = "KFold(5) x 3 seeds"
del _runner, _df_cv

In [ ]:
# Holdout temporal: split 80/20 por publication_date.
# Motivación: detecta distribution shift temporal antes del submit.
# RMSE en USD original (comparable con versiones anteriores).
if temporal_test_idx is not None:
    _X_tt = X.loc[temporal_train_idx]
    _y_tt_fit  = y_fit.loc[temporal_train_idx]
    _y_tt_orig = y_orig.loc[temporal_train_idx]
    _X_th = X.loc[temporal_test_idx]
    _y_th_orig = y_orig.loc[temporal_test_idx]

    _rt = sk.ensemble.RandomForestRegressor(
        n_estimators=n_estimators, max_depth=max_depth,
        n_jobs=-1, random_state=42,
    )
    _rt.fit(_X_tt, _y_tt_fit)
    _pred_th_raw = _rt.predict(_X_th)
    _pred_th = np.exp(_pred_th_raw) if USE_LOG_TARGET else _pred_th_raw
    rmse_holdout_temporal = float(
        sk.metrics.root_mean_squared_error(_y_th_orig, _pred_th)
    )
    print(f"RMSE holdout temporal (80/20 por fecha): {rmse_holdout_temporal:,.2f}")
    EXPERIMENT_LOG["metricas"]["rmse_holdout_temporal"] = rmse_holdout_temporal
    del _rt, _X_tt, _y_tt_fit, _y_tt_orig, _X_th, _y_th_orig, _pred_th_raw, _pred_th
else:
    rmse_holdout_temporal = None
    print("Holdout temporal desactivado (sin publication_date).")
    EXPERIMENT_LOG["metricas"]["rmse_holdout_temporal"] = None

## 3.1. (Opcional) Optimización de hiperparámetros con ResumableRunner

Sweep de `n_estimators` × `max_depth` con 5 folds × 3 seeds por combo (135 fits total ≈ 45-60 min).
Usa `ResumableRunner` para checkpoint append-only: si el proceso cae, la próxima corrida retoma
desde donde quedó.

Para activar: `EXPERIMENT_HP_SWEEP=1 python entregas/run_entrega.py --entrega entrega_3 --nombre v1 ...`

In [ ]:
_SWEEP_ENABLED = os.environ.get("EXPERIMENT_HP_SWEEP", "0") == "1"

if _SWEEP_ENABLED:
    sys.path.insert(0, str(Path("entregas").resolve()))
    from _resumable import ResumableRunner

    _out_dir  = Path(f"entregas/{ENTREGA}")
    _progress = _out_dir / f"hp_sweep_{NOMBRE}.progress.jsonl"
    _out_dir.mkdir(parents=True, exist_ok=True)

    ### Pueden cambiar los hiperparámetros, pero no el modelo
    _grid = [
        {"n_est": n, "max_d": d, "fold": f, "seed": s}
        for n in [500, 1000, 1500]
        for d in [50, 70, None]
        for f in range(5)
        for s in [0, 1, 2]
    ]  # 9 combos × 15 fits = 135 fits

    def _run_one(combo):
        kf = sk.model_selection.KFold(n_splits=5, shuffle=True, random_state=combo["seed"])
        folds = list(kf.split(X, y))
        tr_idx, te_idx = folds[combo["fold"]]
        rf = sk.ensemble.RandomForestRegressor(
            n_estimators=combo["n_est"], max_depth=combo["max_d"],
            n_jobs=-1, random_state=42,
        )
        rf.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = rf.predict(X.iloc[te_idx])
        return {"rmse": float(sk.metrics.root_mean_squared_error(y.iloc[te_idx], pred))}

    runner = ResumableRunner(
        progress_path=_progress,
        grid=_grid,
        run_fn=_run_one,
        key_fn=lambda c: f"n{c['n_est']}_d{str(c['max_d'])}_f{c['fold']}_s{c['seed']}",
        log_every=5,
    )
    runner.run()

    hp_df = runner.results_df()
    _ok = hp_df[hp_df["error"].isna()].copy() if "error" in hp_df.columns else hp_df.copy()
    if not _ok.empty and "rmse" in _ok.columns:
        hp_summary = (
            _ok.groupby(["n_est", "max_d"])["rmse"]
            .agg(["mean", "std", "count"])
            .sort_values("mean")
        )
        display(hp_summary)
        _best = hp_summary.index[0]
        best_n_estimators = int(_best[0])
        best_max_depth    = _best[1]  # puede ser None
        _best_cv  = float(hp_summary.iloc[0]["mean"])
        _best_std = float(hp_summary.iloc[0]["std"])
        print(f"Mejor combo: n_estimators={best_n_estimators}, max_depth={best_max_depth}")
        print(f"RMSE CV5-sweep del mejor combo: {_best_cv:,.2f} ± {_best_std:,.2f}")
        # Actualizar métrica primaria con el mejor HP del sweep
        EXPERIMENT_LOG["metricas"]["rmse_cv5_mean"] = _best_cv
        EXPERIMENT_LOG["metricas"]["rmse_cv5_std"]  = _best_std
        EXPERIMENT_LOG["params"]["best_hp_from_sweep"] = {
            "n_estimators": best_n_estimators, "max_depth": str(best_max_depth),
        }
else:
    best_n_estimators = n_estimators
    best_max_depth    = max_depth
    print(f"HP sweep desactivado. best_n_estimators={best_n_estimators}, best_max_depth={best_max_depth}.")
    print("Para activar: EXPERIMENT_HP_SWEEP=1 python entregas/run_entrega.py ...")

## 3.2. Análisis de la importancia de variables en el modelo (opcional)

Una manera visual de entender a qué variable el modelo le presta mayor atención.

In [ ]:
feat_importances = pd.Series(reg.feature_importances_, index=X.columns)
feat_importances.nlargest(15).plot(kind='barh')
plt.title(f"Feature importances — {NOMBRE}")
plt.tight_layout()
plt.show()

## 4. Solución para subir Kaggle

`df_ap` ya viene preprocesado en paralelo. Entrenamos el modelo final con los mejores HP.

In [ ]:
df_ap.head(2)

In [ ]:
df_ap.shape

In [ ]:
# Entrenamos el modelo final con los mejores HP (del sweep si corrió, o defaults de env)
reg = sk.ensemble.RandomForestRegressor(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    n_jobs=-1,
    random_state=42,
)
reg.fit(X, y_fit)
print(f"Modelo final: n_estimators={best_n_estimators}, max_depth={best_max_depth}")
print(f"Entrenado con {len(X):,} filas y {X.shape[1]} features.")
if USE_LOG_TARGET:
    print("  target: log(price) — predicciones se van a exp() antes del Hot Deck.")

## 4.2. Generación del archivo para Kaggle

In [ ]:
df_ap_num = df_ap.select_dtypes("number").fillna(0)
X_ap      = df_ap_num.reindex(columns=X.columns, fill_value=0)
y_pred_ap_raw = reg.predict(X_ap)
# Si el modelo fue entrenado en log(price), invertimos antes del Hot Deck override
y_pred_ap = np.exp(y_pred_ap_raw) if USE_LOG_TARGET else y_pred_ap_raw
print(f"Predicciones generadas para {len(y_pred_ap)} filas.")
print(pd.Series(y_pred_ap).describe(percentiles=[.05, .5, .95]))

In [ ]:
# 1) Predicción cruda del modelo
df_ap["price"] = y_pred_ap

# 2) Override Hot Deck
n_override = int(df_ap.index.isin(hotdeck_dict).sum())
df_ap.loc[df_ap.index.isin(hotdeck_dict), "price"] = (
    df_ap.loc[df_ap.index.isin(hotdeck_dict)].index.map(hotdeck_dict)
)
print(f"Hot Deck aplicado a {n_override} filas ({n_override/len(df_ap)*100:.1f}%).")

# 3) Redondeo a 1 000 USD (85.6 % de precios en el train son múltiplos de 1 000)
df_ap["price"] = (df_ap["price"] / 1000).round() * 1000

# 4) Sanity check: nada menor a 1 000
df_ap["price"] = df_ap["price"].clip(lower=1000).fillna(df_ap["price"].median())

# 5) Guardar CSV + JSON + leaderboard
out_dir = DIR if IN_COLAB else f"entregas/{ENTREGA}"
os.makedirs(out_dir, exist_ok=True)

stem      = f"solucion-{ENTREGA.replace('_', '')}-{NOMBRE}"
csv_path  = f"{out_dir}/{stem}.csv"
json_path = f"{out_dir}/{stem}.json"

df_ap["price"].to_csv(csv_path)

with open(csv_path, "rb") as _f:
    csv_md5 = hashlib.md5(_f.read()).hexdigest()

EXPERIMENT_LOG["data"]["test_predicciones"]           = int(len(df_ap))
EXPERIMENT_LOG["data"]["hotdeck_overrides_aplicados"] = n_override
EXPERIMENT_LOG["metricas"]["pred_p05"]     = float(df_ap["price"].quantile(0.05))
EXPERIMENT_LOG["metricas"]["pred_mediana"] = float(df_ap["price"].median())
EXPERIMENT_LOG["metricas"]["pred_p95"]     = float(df_ap["price"].quantile(0.95))
EXPERIMENT_LOG["output"] = {
    "csv":      csv_path,
    "csv_md5":  csv_md5,
    "json":     json_path,
    "kaggle_submit_message": f"{ENTREGA}/{NOMBRE}: {DESCRIPCION}",
}

with open(json_path, "w", encoding="utf-8") as _f:
    json.dump(EXPERIMENT_LOG, _f, indent=2, ensure_ascii=False)

# Actualizar leaderboard local (E3 incluye columna RMSE holdout temporal)
if not IN_COLAB:
    lb_path = f"entregas/{ENTREGA}/leaderboard.md"
    header = (
        f"# Leaderboard local — {ENTREGA}\n\n"
        "| nombre | fecha | RMSE CV5 (mean ± std) | RMSE holdout temporal | RMSE holdout | RMSE Kaggle | hotdeck % | descripción | csv md5 |\n"
        "|---|---|---:|---:|---:|---:|---:|---|---|\n"
    )
    if not os.path.exists(lb_path):
        with open(lb_path, "w", encoding="utf-8") as _f:
            _f.write(header)

    cv_mean  = EXPERIMENT_LOG["metricas"].get("rmse_cv5_mean")
    cv_std   = EXPERIMENT_LOG["metricas"].get("rmse_cv5_std")
    temp_val = EXPERIMENT_LOG["metricas"].get("rmse_holdout_temporal")
    hold_val = EXPERIMENT_LOG["metricas"].get("rmse_holdout")
    cv_cell   = f"{cv_mean:,.2f} ± {cv_std:,.0f}" if cv_mean is not None else "—"
    temp_cell = f"{temp_val:,.2f}" if temp_val is not None else "—"
    hold_cell = f"{hold_val:.2f}" if hold_val is not None else "—"
    fila = (
        f"| {NOMBRE} "
        f"| {EXPERIMENT_LOG['timestamp'][:16].replace('T', ' ')} "
        f"| {cv_cell} "
        f"| {temp_cell} "
        f"| {hold_cell} "
        f"| _pendiente_ "
        f"| {EXPERIMENT_LOG['data']['hotdeck_coverage_pct']:.1f}% "
        f"| {DESCRIPCION} "
        f"| `{csv_md5[:8]}` |\n"
    )
    with open(lb_path, "a", encoding="utf-8") as _f:
        _f.write(fila)
    print(f"Leaderboard actualizado: {lb_path}")

print("---")
print(f"CSV : {csv_path}")
print(f"JSON: {json_path}")
print(f"MD5 : {csv_md5}")
print(f"Para Kaggle: {EXPERIMENT_LOG['output']['kaggle_submit_message']}")
df_ap["price"].describe(percentiles=[.05, .5, .95])

## 5. Análisis de los errores

In [ ]:
X_tr_err, X_te_err, y_tr_err, y_te_err = sk.model_selection.train_test_split(
    X, y_orig, test_size=0.2, random_state=42
)
y_fit_err = y_fit.loc[X_tr_err.index]

_re = sk.ensemble.RandomForestRegressor(
    n_estimators=best_n_estimators, max_depth=best_max_depth,
    n_jobs=-1, random_state=42,
)
_re.fit(X_tr_err, y_fit_err)
_pred_raw_err = _re.predict(X_te_err)
y_pred_err = np.exp(_pred_raw_err) if USE_LOG_TARGET else _pred_raw_err

X_te_err = X_te_err.copy()
X_te_err["error"]      = abs(y_pred_err - y_te_err)
X_te_err["price"]      = y_te_err
X_te_err["pred_price"] = y_pred_err
X_te_err.sort_values("error", ascending=False).head(10)